# 02 — Création des features et de la target

On part du dataset nettoyé (`data/processed/btc_clean.csv`) et on crée les colonnes utiles au modèle :

- des **features** = ce que le marché donne à l'instant `t` (présent + passé)
- une **target** = `volatility_24h_future`, ce qu'on veut prédire (futur)

> NB: une **feature** ne regarde que le passé/présent. Seule la **target** a le droit de regarder le futur.

## Partie 1 — Imports & chargement des données

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
CLEAN_PATH = Path("../data/processed/btc_clean.csv")

df = pd.read_csv(CLEAN_PATH, parse_dates=["open_time", "close_time"])

print("Shape :", df.shape)
df.head()

Shape : (8760, 11)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2025-07-01 00:00:00,107146.51,107449.25,107026.93,107377.02,253.62802,2025-07-01 00:59:59.999999,2.720125e+07,68242,114.96074,1.232970e+07
1,2025-07-01 01:00:00,107377.03,107540.00,107171.42,107220.00,219.64653,2025-07-01 01:59:59.999999,2.358859e+07,67588,80.21028,8.614095e+06
2,2025-07-01 02:00:00,107220.00,107410.74,107097.87,107101.69,207.06087,2025-07-01 02:59:59.999999,2.220821e+07,54684,81.41164,8.731958e+06
3,2025-07-01 03:00:00,107101.68,107223.00,107070.20,107192.38,293.12458,2025-07-01 03:59:59.999999,3.141034e+07,38802,152.24013,1.631451e+07
4,2025-07-01 04:00:00,107192.39,107203.11,106867.43,106881.60,261.75371,2025-07-01 04:59:59.999999,2.801826e+07,42983,89.94461,9.629883e+06


In [11]:
print("Colonnes :", df.columns.tolist())
print("\nTypes :")
print(df.dtypes)
print("\nTrie par date croissante :", df["open_time"].is_monotonic_increasing)
print("Doublons open_time :", df.duplicated(subset=["open_time"]).sum())

Colonnes : ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume']

Types :
open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

Trie par date croissante : True
Doublons open_time : 0


## Partie 2 — `return_1h` (le rendement horaire)

C'est la **brique de base** de tout le projet : la variation de prix d'une heure à l'autre.

**Formule :** `return_1h = (close_t - close_t-1) / close_t-1`

Autrement dit : « de combien (en %) le prix de clôture a bougé par rapport à l'heure précédente ».
- `+0.01` = le prix a monté de 1 %
- `-0.02` = le prix a baissé de 2 %

En pandas, `.pct_change()` calcule exactement ça automatiquement. La **1ʳᵉ ligne sera `NaN`** (pas d'heure précédente pour la comparer) — c'est normal, on gérera tous les `NaN` à la fin.

In [12]:
df["return_1h"] = df["close"].pct_change()

df[["open_time", "close", "return_1h"]].head()

,open_time,close,return_1h
0,2025-07-01 00:00:00,107377.02,NaN
1,2025-07-01 01:00:00,107220.00,-0.001462
2,2025-07-01 02:00:00,107101.69,-0.001103
3,2025-07-01 03:00:00,107192.38,0.000847
4,2025-07-01 04:00:00,106881.60,-0.002899


## Partie 3 — Volatilité passée & future (la TARGET)

La **volatilité** = à quel point le prix bouge. On la mesure par l'**écart-type des `return_1h`** sur une fenêtre de 24h.

- `volatility_24h_past` = écart-type des returns sur les **24h passées** (fenêtre `[t-23 ... t]`) → **feature** (info connue à l'instant t)
- `volatility_24h_future` = écart-type des returns sur les **24h suivantes** (fenêtre `[t+1 ... t+24]`) → **TARGET** (ce qu'on veut prédire)

> Astuce pandas : `rolling(24).std()` calcule l'écart-type glissant sur 24 lignes (en regardant **en arrière**). Pour la version *future*, on décale ce résultat de 24 lignes vers le haut avec `.shift(-24)`, ce qui aligne la fenêtre sur le futur. Les deux fenêtres **ne se chevauchent pas** (l'une finit à `t`, l'autre commence à `t+1`) → pas de triche.

In [13]:
df["volatility_24h_past"] = df["return_1h"].rolling(window=24).std()

df["volatility_24h_future"] = df["return_1h"].rolling(window=24).std().shift(-24)

df[["open_time", "return_1h", "volatility_24h_past", "volatility_24h_future"]].head(3)

,open_time,return_1h,volatility_24h_past,volatility_24h_future
0,2025-07-01 00:00:00,NaN,NaN,0.002854
1,2025-07-01 01:00:00,-0.001462,NaN,0.002956
2,2025-07-01 02:00:00,-0.001103,NaN,0.002993


## Partie 4 — Les autres features (amplitude, volume, activité, temps)

On crée tout le reste d'un coup, regroupé par thème :

| Feature | Formule | Ce que ça mesure |
|---|---|---|
| `high_low_range` | `(high - low) / close` | amplitude du prix dans l'heure |
| `open_close_return` | `(close - open) / open` | sens du mouvement dans l'heure |
| `volume_change` | variation du `volume` | accélération/ralentissement de l'activité |
| `quote_volume_change` | variation du `quote_asset_volume` | idem, en valeur (USDT) |
| `trade_intensity` | `number_of_trades / volume` | nb de trades par unité de volume |
| `buy_pressure` | `taker_buy_base_volume / volume` | part des achats « agressifs » |
| `hour` | heure de `open_time` (0–23) | effet heure de la journée |
| `day_of_week` | jour de `open_time` (0=lundi) | effet jour de la semaine |

Toutes ne regardent que le présent/passé → ce sont bien des **features** valides.

In [ ]:
df["high_low_range"] = (df["high"] - df["low"]) / df["close"]
df["open_close_return"] = (df["close"] - df["open"]) / df["open"]

df["volume_change"] = df["volume"].pct_change()
df["quote_volume_change"] = df["quote_asset_volume"].pct_change()


df["trade_intensity"] = df["number_of_trades"] / df["volume"].replace(0, np.nan)
df["buy_pressure"] = df["taker_buy_base_volume"] / df["volume"].replace(0, np.nan)

df["hour"] = df["open_time"].dt.hour
df["day_of_week"] = df["open_time"].dt.dayofweek

df.head(3)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,volatility_24h_past,volatility_24h_future,high_low_range,open_close_return,volume_change,quote_volume_change,trade_intensity,buy_pressure,hour,day_of_week
0,2025-07-01 00:00:00,107146.51,107449.25,107026.93,107377.02,253.62802,2025-07-01 00:59:59.999999,2.720125e+07,68242,114.96074,...,NaN,0.002854,0.003933,0.002151,NaN,NaN,269.063331,0.453265,0,1
1,2025-07-01 01:00:00,107377.03,107540.00,107171.42,107220.00,219.64653,2025-07-01 01:59:59.999999,2.358859e+07,67588,80.21028,...,NaN,0.002956,0.003438,-0.001462,-0.133982,-0.132812,307.712578,0.365179,1,1
2,2025-07-01 02:00:00,107220.00,107410.74,107097.87,107101.69,207.06087,2025-07-01 02:59:59.999999,2.220821e+07,54684,81.41164,...,NaN,0.002993,0.002921,-0.001103,-0.057300,-0.058519,264.096253,0.393177,2,1


## Partie 4 bis — Features avancées de volatilité (multi-horizons)

Le meilleur prédicteur de la volatilité future, c'est la **volatilité passée observée sur plusieurs horizons** (structure dite *HAR* — Corsi 2009). Jusqu'ici on n'avait qu'**une seule** fenêtre de 24h : on ajoute donc plusieurs horizons + des mesures d'amplitude complémentaires.

| Feature | Formule | Ce que ça mesure |
|---|---|---|
| `rv_6h`, `rv_72h`, `rv_168h` | écart-type des returns sur 6h / 72h / 168h | volatilité passée à court / moyen / long terme |
| `parkinson` | estimateur de volatilité basé sur high/low | volatilité intra-heure (plus précis) |
| `atr_14` | amplitude moyenne sur 14h / close | amplitude récente typique |
| `vol_ma_ratio` | `volume / moyenne(volume, 24h)` | pic d'activité par rapport à la normale |

Toutes ne regardent que le passé/présent → ce sont des **features** valides (pas de triche).

In [15]:
# Volatilite passee sur plusieurs horizons (court / moyen / long terme)
df["rv_6h"] = df["return_1h"].rolling(window=6).std()
df["rv_72h"] = df["return_1h"].rolling(window=72).std()
df["rv_168h"] = df["return_1h"].rolling(window=168).std()

# Estimateur de Parkinson (volatilite basee sur high/low)
df["parkinson"] = np.sqrt((1 / (4 * np.log(2))) * (np.log(df["high"] / df["low"]) ** 2))

# Amplitude moyenne recente (ATR normalise) et pic de volume
df["atr_14"] = (df["high"] - df["low"]).rolling(window=14).mean() / df["close"]
df["vol_ma_ratio"] = df["volume"] / df["volume"].rolling(window=24).mean()

df[["rv_6h", "rv_72h", "rv_168h", "parkinson", "atr_14", "vol_ma_ratio"]].head(3)

,rv_6h,rv_72h,rv_168h,parkinson,atr_14,vol_ma_ratio
0,NaN,NaN,NaN,0.002365,NaN,NaN
1,NaN,NaN,NaN,0.002062,NaN,NaN
2,NaN,NaN,NaN,0.001752,NaN,NaN


## Partie 5 — Gestion des NaN & sauvegarde

Certaines features créent des `NaN` :
- au **début** : `return_1h` (1 ligne), `volatility_24h_past` (23 lignes), `volume_change`… → pas assez d'historique
- à la **fin** : `volatility_24h_future` (24 dernières lignes) → pas assez de futur

On **supprime ces lignes incomplètes** (`dropna`) pour ne garder que les lignes où toutes les colonnes existent, puis on sauvegarde le dataset enrichi dans `data/processed/btc_features.csv`.

In [16]:
print("NaN par colonne :")
print(df.isna().sum())

lignes_avant = len(df)
df_features = df.dropna().reset_index(drop=True)
print(f"\nLignes : {lignes_avant} -> {len(df_features)} ({lignes_avant - len(df_features)} supprimees)")

OUT_PATH = Path("../data/processed/btc_features.csv")
df_features.to_csv(OUT_PATH, index=False)
print(f"\nDataset enrichi sauvegarde -> {OUT_PATH}")
print("Shape finale :", df_features.shape)
df_features.head(3)

NaN par colonne :
open_time                   0
open                        0
high                        0
low                         0
close                       0
volume                      0
close_time                  0
quote_asset_volume          0
number_of_trades            0
taker_buy_base_volume       0
taker_buy_quote_volume      0
return_1h                   1
volatility_24h_past        24
volatility_24h_future      24
high_low_range              0
open_close_return           0
volume_change               1
quote_volume_change         1
trade_intensity             0
buy_pressure                0
hour                        0
day_of_week                 0
rv_6h                       6
rv_72h                     72
rv_168h                   168
parkinson                   0
atr_14                     13
vol_ma_ratio               23
dtype: int64

Lignes : 8760 -> 8568 (192 supprimees)

Dataset enrichi sauvegarde -> ../data/processed/btc_features.csv
Shape finale : (8568, 2

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,trade_intensity,buy_pressure,hour,day_of_week,rv_6h,rv_72h,rv_168h,parkinson,atr_14,vol_ma_ratio
0,2025-07-08 00:00:00,108262.94,108525.0,108193.48,108299.99,409.64154,2025-07-08 00:59:59.999999,4.438003e+07,97847,225.69090,...,238.860053,0.550947,0,1,0.001748,0.001661,0.002504,0.001837,0.003783,1.028240
1,2025-07-08 01:00:00,108299.99,108300.0,107689.64,107705.03,327.96475,2025-07-08 01:59:59.999999,3.541732e+07,86676,121.90375,...,264.284500,0.371698,1,1,0.002991,0.001768,0.002538,0.003394,0.004057,0.820791
2,2025-07-08 02:00:00,107705.03,107945.0,107429.57,107766.20,1090.68261,2025-07-08 02:59:59.999999,1.174496e+08,133423,314.56254,...,122.329813,0.288409,2,1,0.002970,0.001767,0.002537,0.002875,0.004172,2.554648
